# FinBERT Earnings Signal Fine-Tuner

Run cells top to bottom with **Shift+Enter**.

In [ ]:
# ── 0. CONFIG ──────────────────────────────────────────────
API_KEY = "74TBXZO1PQNBIN7P"

TICKERS = [
    "AAPL", "MSFT", "NVDA", "GOOGL", "META",
    "AMZN", "TSLA", "JPM", "BAC", "GS",
    "LLY", "JNJ", "UNH", "XOM", "CVX",
    "HD", "WMT", "MCD", "NKE", "CRM"
]

BUY_THRESHOLD  =  0.03
SELL_THRESHOLD = -0.03
MODEL_NAME     = "ProsusAI/finbert"
MAX_TOKENS     = 512
EPOCHS         = 4
BATCH_SIZE     = 8
LEARNING_RATE  = 2e-5
TEST_SPLIT     = 0.15
RANDOM_SEED    = 42

LABEL2ID = {"Buy": 0, "Hold": 1, "Sell": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [ ]:
# ── 1. INSTALL DEPENDENCIES ────────────────────────────────
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers", "datasets", "accelerate",
    "scikit-learn", "pandas", "requests", "tqdm", "seaborn", "matplotlib"])
print("Done.")

In [ ]:
# ── 2. IMPORTS ─────────────────────────────────────────────
import requests, time
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# ── 3. FETCH TRANSCRIPTS ───────────────────────────────────
def fetch_transcripts(ticker):
    url = (f"https://www.alphavantage.co/query"
           f"?function=EARNINGS_CALL_TRANSCRIPT&symbol={ticker}&apikey={API_KEY}")
    data = requests.get(url, timeout=20).json()
    results = []
    for item in data.get("transcript", []):
        text = item.get("content", "")
        date = item.get("date", "")
        if text and date:
            results.append({"ticker": ticker, "date": date,
                            "quarter": item.get("quarter", ""), "text": text})
    return results

print("Fetching transcripts (~5 min due to rate limits)...")
all_transcripts = []
for i, ticker in enumerate(tqdm(TICKERS)):
    try:
        r = fetch_transcripts(ticker)
        all_transcripts.extend(r)
        print(f"  {ticker}: {len(r)} transcripts")
    except Exception as e:
        print(f"  {ticker}: ERROR - {e}")
    if i < len(TICKERS) - 1:
        time.sleep(13)

df_transcripts = pd.DataFrame(all_transcripts)
print(f"\nTotal: {len(df_transcripts)} transcripts")
df_transcripts.head()

In [ ]:
# ── 4. PRICE DATA & LABELS ─────────────────────────────────
_price_cache = {}

def get_daily_prices(ticker):
    if ticker in _price_cache:
        return _price_cache[ticker]
    url = (f"https://www.alphavantage.co/query"
           f"?function=TIME_SERIES_DAILY_ADJUSTED&symbol={ticker}&outputsize=full&apikey={API_KEY}")
    raw = requests.get(url, timeout=20).json().get("Time Series (Daily)", {})
    if not raw:
        return pd.DataFrame()
    df = pd.DataFrame(raw).T
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df["close"] = df["5. adjusted close"].astype(float)
    _price_cache[ticker] = df[["close"]]
    return df[["close"]]

def get_3day_return(ticker, earnings_date):
    prices = get_daily_prices(ticker)
    if prices.empty:
        return None
    future = prices[prices.index >= pd.to_datetime(earnings_date)]
    if len(future) < 4:
        return None
    return (future.iloc[3]["close"] - future.iloc[0]["close"]) / future.iloc[0]["close"]

def return_to_label(ret):
    if ret >= BUY_THRESHOLD:  return "Buy"
    if ret <= SELL_THRESHOLD: return "Sell"
    return "Hold"

print("Fetching price data (~5 min)...")
unique_tickers = df_transcripts["ticker"].unique().tolist()
for i, ticker in enumerate(tqdm(unique_tickers)):
    try:
        get_daily_prices(ticker)
    except Exception as e:
        print(f"  {ticker}: {e}")
    if i < len(unique_tickers) - 1:
        time.sleep(13)

returns, labels = [], []
for _, row in df_transcripts.iterrows():
    ret = get_3day_return(row["ticker"], row["date"])
    returns.append(ret)
    labels.append(return_to_label(ret) if ret is not None else None)

df_transcripts["return_3d"] = returns
df_transcripts["label"] = labels
df_labeled = df_transcripts.dropna(subset=["label", "text"]).copy()
df_labeled["label_id"] = df_labeled["label"].map(LABEL2ID)

print(f"\nLabeled: {len(df_labeled)} samples")
print(df_labeled["label"].value_counts())

In [ ]:
# ── 5. TOKENIZE & BUILD DATASETS ───────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_first_chunk(text):
    return tokenizer(text, max_length=MAX_TOKENS, truncation=True,
                     padding="max_length", return_tensors="pt")

class TranscriptDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = [tokenize_first_chunk(t) for t in texts]
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v.squeeze(0) for k, v in self.encodings[idx].items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

X_train, X_test, y_train, y_test = train_test_split(
    df_labeled["text"].tolist(), df_labeled["label_id"].tolist(),
    test_size=TEST_SPLIT, stratify=df_labeled["label_id"].tolist(),
    random_state=RANDOM_SEED
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print("Tokenizing...")
train_dataset = TranscriptDataset(X_train, y_train)
test_dataset  = TranscriptDataset(X_test,  y_test)
print("Done.")

In [ ]:
# ── 6. LOAD MODEL ──────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
    ignore_mismatched_sizes=True
).to(device)

print(f"Loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── 7. FINE-TUNE ────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    from sklearn.metrics import f1_score, accuracy_score
    return {
        "accuracy":    accuracy_score(labels, preds),
        "f1_macro":    f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

training_args = TrainingArguments(
    output_dir="./finbert-earnings",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=10,
    fp16=False,
    report_to="none",
    seed=RANDOM_SEED,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=test_dataset,
    tokenizer=tokenizer, compute_metrics=compute_metrics,
)

print("Starting fine-tuning (~10-15 min on M2)...")
trainer.train()

In [ ]:
# ── 8. EVALUATE ─────────────────────────────────────────────
predictions = trainer.predict(test_dataset)
preds  = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("\n── Classification Report ──")
print(classification_report(labels, preds, target_names=[ID2LABEL[i] for i in range(3)]))

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=list(LABEL2ID.keys()),
            yticklabels=list(LABEL2ID.keys()), ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix - FinBERT Earnings Signals")
plt.tight_layout()
plt.show()

In [ ]:
# ── 9. SAVE MODEL ───────────────────────────────────────────
SAVE_PATH = "./finbert-earnings-final"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

In [ ]:
# ── 10. INFERENCE HELPER ────────────────────────────────────
def predict_signal(transcript_text):
    model.eval()
    tokens = tokenizer.encode(transcript_text, add_special_tokens=False)
    chunk_size = MAX_TOKENS - 2
    chunks = [tokens[i:i+chunk_size] for i in range(0, len(tokens), chunk_size)]
    all_logits = []
    with torch.no_grad():
        for chunk in chunks:
            ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
            padding = [tokenizer.pad_token_id] * (MAX_TOKENS - len(ids))
            input_ids      = torch.tensor([ids + padding]).to(device)
            attention_mask = torch.tensor([[1]*len(ids) + [0]*len(padding)]).to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            all_logits.append(logits.cpu())
    probs = torch.softmax(torch.stack(all_logits).mean(dim=0), dim=-1).squeeze().numpy()
    pred_id = int(probs.argmax())
    return {
        "signal":     ID2LABEL[pred_id],
        "confidence": float(probs[pred_id]),
        "scores": {"Buy": float(probs[0]), "Hold": float(probs[1]), "Sell": float(probs[2])}
    }

result = predict_signal(X_test[0])
print(f"Predicted: {result['signal']} ({result['confidence']:.1%})")
print(f"Actual:    {ID2LABEL[y_test[0]]}")
print(f"Scores:    {result['scores']}")